In [7]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import SpectralClustering
from sklearn.decomposition import PCA

import plotly.express as px


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

INPUT_FILE = "../data_clean/fighter_career_stats.csv"

K_CLUSTERS = 12
MIN_FIGHTS = 4
N_NEIGHBORS = 15
RANDOM_STATE = 100


# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

df_model = df[df["total_fights"] >= MIN_FIGHTS].copy()

print("Original fighters:", len(df))
print("Fighters after min-fight filter:", len(df_model))


# ------------------------------------------------------------
# 2. Select clustering features
# ------------------------------------------------------------

exclude_cols = [
    "fighter",            # ID column
    "total_fights",       # sample-size / reliability column
    "total_fight_time"    # career-volume column
]

opponent_stat_patterns = [
    #"_absorbed_",         # damage/offense received from opponents
   # "_opp_att_"           # opponent attempt volume
]

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]

feature_cols = [
    col for col in feature_cols
    if pd.api.types.is_numeric_dtype(df_model[col])
    and not any(pattern in col for pattern in opponent_stat_patterns)
]

excluded_opponent_cols = [
    col for col in df_model.columns
    if any(pattern in col for pattern in opponent_stat_patterns)
]

print("Number of clustering features:", len(feature_cols))
print("Excluded opponent-related features:", excluded_opponent_cols)


# ------------------------------------------------------------
# 3. Feature weighting
# ------------------------------------------------------------

HIGH_WEIGHT = 2.0
MEDIUM_WEIGHT = 1.0
LOW_WEIGHT = 0.5


def get_feature_weight(feature_name):
    """
    Apply style-priority weights after normalization.
    """

    # Low weight: opponent activity, damage absorbed, and loss methods.
    if (
        "_absorbed_" in feature_name
        or "_opp_att_" in feature_name
        or feature_name.startswith("loss_")
    ):
        return LOW_WEIGHT

    # High weight: win methods, control, knockdowns, submissions, and fighter activity.
    if (
        feature_name.startswith("win_")
        or feature_name in ["CTRL_pct", "KD_per_min", "SUB_ATT_per_min"]
        or feature_name.endswith("_att_per_min")
        or feature_name.endswith("_land_per_min")
    ):
        return HIGH_WEIGHT

    # Medium weight: pace, accuracy, defense, and reversals.
    if feature_name in [
        "fight_quickness",
        "TOTAL_STR_acc",
        "SIG_STR_acc",
        "TD_acc",
        "SIG_STR_def",
        "TOTAL_STR_def",
        "TD_def",
        "REV_per_min",
    ]:
        return MEDIUM_WEIGHT

    return MEDIUM_WEIGHT


feature_weights = pd.Series(
    {feature: get_feature_weight(feature) for feature in feature_cols},
    name="weight"
)

print("\nFeature weight counts:")
print(feature_weights.value_counts().sort_index())


# ------------------------------------------------------------
# 4. Preprocess features
# ------------------------------------------------------------

X_raw = df_model[feature_cols].replace([np.inf, -np.inf], np.nan)

preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

X_scaled = preprocess.fit_transform(X_raw)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=feature_cols,
    index=df_model.index
)

X_weighted_df = X_scaled_df.mul(feature_weights, axis=1)
X_weighted = X_weighted_df.to_numpy()


# ------------------------------------------------------------
# 5. Spectral clustering
# ------------------------------------------------------------

spectral = SpectralClustering(
    n_clusters=K_CLUSTERS,
    affinity="nearest_neighbors",
    n_neighbors=N_NEIGHBORS,
    assign_labels="kmeans",
    random_state=RANDOM_STATE
)

df_model["style_cluster"] = spectral.fit_predict(X_weighted)


# ------------------------------------------------------------
# 6. Cluster counts and summary
# ------------------------------------------------------------

cluster_counts = df_model["style_cluster"].value_counts().sort_index()

cluster_summary = (
    df_model
    .groupby("style_cluster")[feature_cols]
    .mean()
    .round(3)
)

cluster_summary_table = cluster_summary.T
cluster_summary_table.index.name = "feature"

cluster_summary_table.columns = [
    f"Cluster {col}" for col in cluster_summary_table.columns
]

cluster_summary_scaled = (
    X_weighted_df
    .groupby(df_model["style_cluster"])
    .mean()
    .round(3)
)

cluster_summary_scaled_table = cluster_summary_scaled.T
cluster_summary_scaled_table.index.name = "feature"

cluster_summary_scaled_table.columns = [
    f"Cluster {col}" for col in cluster_summary_scaled_table.columns
]

feature_display_order = (
    feature_weights
    .sort_values(ascending=False, kind="mergesort")
    .index
)

cluster_summary_scaled_table = cluster_summary_scaled_table.loc[feature_display_order]

print("\nCluster counts:")
print(cluster_counts)

print("\nWeighted normalized cluster profile means used by clustering:")
display(cluster_summary_scaled_table)






Original fighters: 2683
Fighters after min-fight filter: 1560
Number of clustering features: 53
Excluded opponent-related features: []

Feature weight counts:
weight
0.5    21
1.0     8
2.0    24
Name: count, dtype: int64

Cluster counts:
style_cluster
0     133
1      51
2     190
3     149
4     257
5      60
6      69
7      49
8     131
9     134
10     90
11    247
Name: count, dtype: int64

Weighted normalized cluster profile means used by clustering:


,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8,Cluster 9,Cluster 10,Cluster 11
win_DEC_pct,0.707,-1.451,0.642,-1.151,0.421,-1.603,0.644,0.525,-1.278,0.171,-0.625,-0.359
win_KO_TKO_pct,-0.205,2.100,-0.441,-0.490,0.184,-0.548,0.010,0.203,1.582,0.290,0.504,0.517
win_SUB_pct,0.679,0.471,1.145,3.824,0.360,4.667,0.455,0.306,0.336,0.714,1.627,1.034
CTRL_pct,1.016,-0.643,1.926,-0.029,-0.735,0.438,-0.073,-0.939,-0.740,-0.667,2.513,0.532
KD_per_min,-0.176,5.209,-0.357,-0.028,0.355,-0.516,-0.177,0.305,1.380,0.619,0.547,0.532
SUB_ATT_per_min,-0.037,-0.153,0.702,1.705,-0.438,5.065,-0.103,-0.492,0.097,-0.204,1.799,0.554
SIG_STR_land_per_min,0.818,2.303,-1.409,-0.494,0.397,-2.059,1.676,3.393,-1.164,1.807,0.187,-0.102
SIG_STR_att_per_min,0.909,1.877,-1.437,-0.190,0.665,-2.065,1.270,3.577,-1.086,1.511,-0.335,-0.385
TOTAL_STR_land_per_min,1.072,1.284,-0.585,-0.596,-0.283,-1.267,1.895,2.385,-1.511,1.219,1.535,0.328
TOTAL_STR_att_per_min,1.219,1.508,-1.065,-0.303,0.293,-1.858,1.601,3.380,-1.461,1.353,0.540,-0.162


In [37]:
# ------------------------------------------------------------
# 5. Add readable cluster style labels
# ------------------------------------------------------------

cluster_style_names = {
    0: "Balanced Scrambler / Generalist",
    1: "Ground-and-Pound Wrestler",
    2: "Control Submission Grappler",
    3: "High-Pace Brawler",
    4: "Volume Body/Leg Striker",
    5: "Defensively Vulnerable Finisher",
    6: "Pressure Fighter / Damage Sponge",
    7: "Power Striker",
    8: "Clinch War Fighter",
    9: "Defensive Distance Striker",
    10: "Dominant Ground Control Fighter",
    11: "Submission Specialist",
}

df_model["style_label"] = df_model["style_cluster"].map(cluster_style_names)
# ------------------------------------------------------------
# 6. 3D PCA visualization with Plotly
# ------------------------------------------------------------

pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    "fighter": df_model["fighter"].values,
    "style_cluster": df_model["style_cluster"].values,
    "style_label": df_model["style_label"].values,
    "total_fights": df_model["total_fights"].values,
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "PC3": X_pca[:, 2]
})

fig = px.scatter_3d(
    plot_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="style_label",
    hover_name="fighter",
    hover_data={
        "style_cluster": True,
        "total_fights": True,
        "PC1": ":.3f",
        "PC2": ":.3f",
        "PC3": ":.3f"
    },
    title=f"3D PCA Visualization of Spectral Fighter Style Clusters, k={K_CLUSTERS}"
)

fig = px.scatter_3d(
    plot_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="style_label",
    hover_name="fighter",
    hover_data={
        "style_cluster": True,
        "total_fights": True,
        "PC1": ":.3f",
        "PC2": ":.3f",
        "PC3": ":.3f"
    },
    title=f"3D PCA Visualization of Spectral Fighter Style Clusters, k={K_CLUSTERS}",
    width=1000,
    height=800
)

fig.update_traces(
    marker=dict(
        size=3,
        opacity=1.0
    )
)

fig.show(
    config={
        "scrollZoom": True,
        "displayModeBar": True
    }
)

In [9]:
def get_fighter_style(fighter_name, df_model):
    """
    Search for a fighter and return their assigned cluster/style.
    Allows partial, case-insensitive search.
    """

    matches = df_model[
        df_model["fighter"]
        .str.lower()
        .str.contains(fighter_name.lower(), na=False)
    ]

    if matches.empty:
        print(f"No fighter found matching: {fighter_name}")
        return None

    result = matches[
        [
            "fighter",
            "style_cluster",
            "style_label",
            "total_fights"
        ]
    ].sort_values("fighter")

    return result

In [39]:
get_fighter_style("Topuria", df_model)


,fighter,style_cluster,style_label,total_fights
1009,Ilia Topuria,7,Power Striker,9


In [57]:
# ------------------------------------------------------------
# 6. Save fighter style lookup
# ------------------------------------------------------------

fighter_styles = df_model[["fighter", "style_cluster"]].copy()

fighter_styles["style_label"] = fighter_styles["style_cluster"].map(cluster_style_names)

fighter_styles = fighter_styles.sort_values("fighter").reset_index(drop=True)

print("\nFighter style lookup:")
display(fighter_styles.head())



Fighter style lookup:


,fighter,style_cluster,style_label
0,AJ Dobson,1,Defensive Distance Striker
1,AJ Fletcher,1,Defensive Distance Striker
2,Aaron Phillips,4,High-Risk Early-Finish Grappler
3,Aaron Riley,0,Clinch Pressure Fighter
4,Aaron Rosa,0,Clinch Pressure Fighter


Saved: fighter_style_clusters.csv
